# ResNeXt-29 + SLA+SD (paper loss: plain cross-entropy, beta = 1) + ResizeMix

Same recipe as the 0.90 run, with three changes:
1. plain cross-entropy (no label smoothing), as in the SLA paper
2. beta = 1: extra cross-entropy on the 10-class head (SLA paper, eq. 5)
3. ResizeMix instead of CutMix: the whole partner image is shrunk and pasted, so its object is always visible (Qin et al. 2020)

Needs the cache and split already on Drive (`data/cache/*.npy`, `data/splits/split_seed42_...npz`). Run all. If Colab disconnects, Run all again: training resumes from `last.pt`.

## 1. Config

In [1]:
CFG = dict(
    RUN_NAME="resnext29_slasd_plainCE_beta1_resizemix_v1",
    DRIVE_PROJECT_DIR="cifar10_resnet_challenge",
    SPLIT_FILE="split_seed42_val0.05_test0.05_n50000.npz",
    SEED=14,

    # model: ResNeXt-29, 4 groups x 24d, widths 280/528/1040 + SLA head = 4,998,738 params
    CARDINALITY=4, BOTTLENECK_WIDTH=24, WIDTHS=(280, 528, 1040), BLOCKS=(3, 3, 3),

    # training
    EPOCHS=200, BATCH_SIZE=256, LR=0.2, MOMENTUM=0.9, WEIGHT_DECAY=5e-4, WARMUP_EPOCHS=5,
    ASAM_RHO=0.5, EMA_DECAY=0.999,

    # SLA+SD loss (paper eq. 5): joint CE + KL(aggregated || 10-class head) + BETA * CE(10-class head)
    BETA=1.0,

    # augmentation
    TA_GROUP=16, GRAYSCALE_P=0.1, CUTOUT=8,
    MIX_PROB=0.5,                      # share of batches that get ResizeMix
    RESIZEMIX_SCALE=(0.1, 0.8),        # pasted image side = scale * 32 (ResizeMix paper default range)
)
CFG

{'RUN_NAME': 'resnext29_slasd_plainCE_beta1_resizemix_v1',
 'DRIVE_PROJECT_DIR': 'cifar10_resnet_challenge',
 'SPLIT_FILE': 'split_seed42_val0.05_test0.05_n50000.npz',
 'SEED': 14,
 'CARDINALITY': 4,
 'BOTTLENECK_WIDTH': 24,
 'WIDTHS': (280, 528, 1040),
 'BLOCKS': (3, 3, 3),
 'EPOCHS': 200,
 'BATCH_SIZE': 256,
 'LR': 0.2,
 'MOMENTUM': 0.9,
 'WEIGHT_DECAY': 0.0005,
 'WARMUP_EPOCHS': 5,
 'ASAM_RHO': 0.5,
 'EMA_DECAY': 0.999,
 'BETA': 1.0,
 'TA_GROUP': 16,
 'GRAYSCALE_P': 0.1,
 'CUTOUT': 8,
 'MIX_PROB': 0.5,
 'RESIZEMIX_SCALE': (0.1, 0.8)}

## 2. Setup

In [2]:
import os, json, time, math
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torchvision.transforms import v2 as T
from torch.optim.swa_utils import AveragedModel, get_ema_multi_avg_fn
from google.colab import drive

drive.mount("/content/drive")
ROOT = os.path.join("/content/drive/MyDrive", CFG["DRIVE_PROJECT_DIR"])
CACHE = os.path.join(ROOT, "data", "cache")
RUN_DIR = os.path.join(ROOT, "runs", CFG["RUN_NAME"]); os.makedirs(RUN_DIR, exist_ok=True)
LAST, BEST = os.path.join(RUN_DIR, "last.pt"), os.path.join(RUN_DIR, "best.pt")

device = torch.device("cuda")
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True; torch.backends.cudnn.allow_tf32 = True
torch.manual_seed(CFG["SEED"]); np.random.seed(CFG["SEED"])
print("GPU:", torch.cuda.get_device_name(0), "| run dir:", RUN_DIR)

def save(obj, path):                      # write to a temp file first so a disconnect never leaves a broken checkpoint
    torch.save(obj, path + ".tmp"); os.replace(path + ".tmp", path)

CLASSES = ["airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]

Mounted at /content/drive
GPU: NVIDIA L4 | run dir: /content/drive/MyDrive/cifar10_resnet_challenge/runs/resnext29_slasd_plainCE_beta1_resizemix_v1


## 3. Data (45k train / 2.5k val / 2.5k internal test, same split as every earlier run)

In [3]:
X = np.load(os.path.join(CACHE, "train_x.npy")); y = np.load(os.path.join(CACHE, "train_y.npy"))
X_kaggle = np.load(os.path.join(CACHE, "test_x.npy")); kaggle_ids = np.load(os.path.join(CACHE, "test_ids.npy"))
sample_sub = pd.read_csv(os.path.join(CACHE, "sample_submission.csv"))
split = np.load(os.path.join(ROOT, "data", "splits", CFG["SPLIT_FILE"]))

def gpu(a): return torch.from_numpy(a).permute(0, 3, 1, 2).contiguous().to(device)      # N,3,32,32 uint8
Xtr, ytr = gpu(X[split["train"]]), torch.from_numpy(y[split["train"]]).to(device)
Xva, yva = gpu(X[split["val"]]), torch.from_numpy(y[split["val"]]).to(device)
Xte, yte = gpu(X[split["test"]]), torch.from_numpy(y[split["test"]]).to(device)
Xkg = gpu(X_kaggle)

MEAN = (Xtr.float() / 255).mean((0, 2, 3)).view(1, 3, 1, 1)
STD = (Xtr.float() / 255).std((0, 2, 3)).view(1, 3, 1, 1)
def normalize(x): return ((x.float() / 255 - MEAN) / STD).contiguous(memory_format=torch.channels_last)
print("train", len(Xtr), "| val", len(Xva), "| internal test", len(Xte), "| kaggle", len(Xkg))

train 45000 | val 2500 | internal test 2500 | kaggle 2000


## 4. Augmentation: TrivialAugment, grayscale, crop + flip, cutout, ResizeMix

In [4]:
trivial = T.TrivialAugmentWide()

def augment(x_u8):
    n, ar = len(x_u8), torch.arange(32, device=device)
    x_u8 = torch.cat([trivial(c) for c in x_u8.split(CFG["TA_GROUP"])])      # one random op per group of images
    gray = (x_u8.float() * torch.tensor([0.299, 0.587, 0.114], device=device).view(1, 3, 1, 1)).sum(1, keepdim=True)
    g = torch.rand(n, device=device) < CFG["GRAYSCALE_P"]
    x_u8 = torch.where(g[:, None, None, None], gray.round().to(torch.uint8).expand(-1, 3, -1, -1), x_u8)
    x = (x_u8.float() / 255 - MEAN) / STD
    xp = F.pad(x, (4, 4, 4, 4), mode="reflect")                               # random crop with 4px padding
    dy, dx = torch.randint(0, 9, (n,), device=device), torch.randint(0, 9, (n,), device=device)
    x = xp[torch.arange(n, device=device)[:, None, None, None], torch.arange(3, device=device)[None, :, None, None],
           (dy[:, None] + ar)[:, None, :, None], (dx[:, None] + ar)[:, None, None, :]]
    flip = torch.rand(n, device=device) < 0.5
    x = torch.where(flip[:, None, None, None], x.flip(3), x)
    cy, cx, h = torch.randint(0, 32, (n,), device=device), torch.randint(0, 32, (n,), device=device), CFG["CUTOUT"] // 2
    hole = ((ar[None, :, None] - cy[:, None, None]).abs() < h) & ((ar[None, None, :] - cx[:, None, None]).abs() < h)
    return (x * (~hole)[:, None].float()).contiguous(memory_format=torch.channels_last)

def resizemix(x, y):
    """ResizeMix (Qin et al. 2020): shrink a shuffled partner image and paste it whole at a random spot.
    Label weight of the partner = pasted area."""
    idx = torch.randperm(len(x), device=device)
    s = max(2, int(32 * np.random.uniform(*CFG["RESIZEMIX_SCALE"])))
    r, c = np.random.randint(0, 33 - s), np.random.randint(0, 33 - s)
    x = x.clone()
    x[:, :, r:r + s, c:c + s] = F.interpolate(x[idx], size=(s, s), mode="bilinear", align_corners=False)
    lam = 1 - s * s / 1024
    return x.contiguous(memory_format=torch.channels_last), y, y[idx], lam

## 5. Model: ResNeXt-29 with an extra SLA head (class x rotation, used only in training)

In [5]:
class Block(nn.Module):
    def __init__(self, cin, inner, cout, stride, groups):
        super().__init__()
        self.c1, self.b1 = nn.Conv2d(cin, inner, 1, bias=False), nn.BatchNorm2d(inner)
        self.c2, self.b2 = nn.Conv2d(inner, inner, 3, stride, 1, groups=groups, bias=False), nn.BatchNorm2d(inner)
        self.c3, self.b3 = nn.Conv2d(inner, cout, 1, bias=False), nn.BatchNorm2d(cout)
        nn.init.zeros_(self.b3.weight)
        self.skip = nn.Identity() if stride == 1 and cin == cout else nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=False), nn.BatchNorm2d(cout))
    def forward(self, x):
        out = F.relu(self.b1(self.c1(x)), inplace=True)
        out = F.relu(self.b2(self.c2(out)), inplace=True)
        return F.relu(self.b3(self.c3(out)) + self.skip(x), inplace=True)

class ResNeXt29(nn.Module):
    def __init__(self, groups, width, widths, blocks):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3, 64, 3, 1, 1, bias=False), nn.BatchNorm2d(64), nn.ReLU(inplace=True))
        layers, cin = [], 64
        for i, (cout, nb) in enumerate(zip(widths, blocks)):
            for j in range(nb):
                layers.append(Block(cin, groups * width * 2 ** i, cout, 2 if (j == 0 and i > 0) else 1, groups)); cin = cout
        self.layers = nn.Sequential(*layers)
        self.fc = nn.Linear(cin, 10)              # 10-class head: used for the submission
        self.fc_joint = nn.Linear(cin, 40)        # SLA head: 10 classes x 4 rotations, training only
        for m in self.modules():
            if isinstance(m, nn.Conv2d): nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
    def features(self, x):
        return torch.flatten(F.adaptive_avg_pool2d(self.layers(self.stem(x)), 1), 1)
    def forward(self, x):
        return self.fc(self.features(x))
    def both(self, x):
        f = self.features(x)
        return self.fc(f), self.fc_joint(f)

model = ResNeXt29(CFG["CARDINALITY"], CFG["BOTTLENECK_WIDTH"], CFG["WIDTHS"], CFG["BLOCKS"]).to(device).to(memory_format=torch.channels_last)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"trainable parameters: {n_params:,}")
assert n_params <= 5_000_000

trainable parameters: 4,998,738


## 6. Optimizer (SGD Nesterov), warm-up + cosine schedule, EMA, prediction

In [6]:
decay = [p for p in model.parameters() if p.ndim > 1]
no_decay = [p for p in model.parameters() if p.ndim <= 1]          # no weight decay on BatchNorm and biases
opt = torch.optim.SGD([{"params": decay, "weight_decay": CFG["WEIGHT_DECAY"]}, {"params": no_decay, "weight_decay": 0.0}],
                      lr=CFG["LR"], momentum=CFG["MOMENTUM"], nesterov=True)
steps = math.ceil(len(Xtr) / CFG["BATCH_SIZE"]); warm = CFG["WARMUP_EPOCHS"] * steps
sched = torch.optim.lr_scheduler.SequentialLR(opt, [
    torch.optim.lr_scheduler.LinearLR(opt, start_factor=0.01, total_iters=warm),
    torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG["EPOCHS"] * steps - warm)], milestones=[warm])
ema = AveragedModel(model, multi_avg_fn=get_ema_multi_avg_fn(CFG["EMA_DECAY"]), use_buffers=True)

@torch.no_grad()
def predict(m, X, mode="sd", tta=True):
    """Class probabilities. sd: 10-class head | ag: average of the SLA head over 4 rotations | sdag: mean of both.
    tta: also average over the horizontally flipped image."""
    m.eval(); out = []
    def probs(xb):
        if mode == "sd": return m(xb).float().softmax(1)
        ag = sum(m.both(torch.rot90(xb, k, (2, 3)))[1].float().view(-1, 10, 4)[:, :, k] for k in range(4)) / 4
        return ag.softmax(1) if mode == "ag" else 0.5 * (ag.softmax(1) + m(xb).float().softmax(1))
    for i in range(0, len(X), 1000):
        xb = normalize(X[i:i + 1000])
        with torch.autocast("cuda", dtype=torch.bfloat16):
            p = probs(xb)
            if tta: p = (p + probs(xb.flip(3))) / 2
        out.append(p)
    return torch.cat(out)

def acc(p, y): return (p.argmax(1) == y).float().mean().item()

## 7. Train (resumes from last.pt)

In [7]:
start, best_acc, history = 0, 0.0, []
if os.path.exists(LAST):
    ck = torch.load(LAST, map_location=device, weights_only=False)
    assert ck["cfg"] == CFG, "Config changed since this run started: use a new RUN_NAME"
    model.load_state_dict(ck["model"]); ema.load_state_dict(ck["ema"])
    opt.load_state_dict(ck["opt"]); sched.load_state_dict(ck["sched"])
    start, best_acc, history = ck["epoch"], ck["best_acc"], ck["history"]
    print(f"resumed at epoch {start}, best val {best_acc:.4f}")

params = [p for p in model.parameters()]
rot = torch.arange(4, device=device)

def loss_fn(x4, ya, yb, lam, B):
    """SLA+SD (paper eq. 5) with plain cross-entropy, on 4 rotated copies of the batch."""
    with torch.autocast("cuda", dtype=torch.bfloat16):
        sd_lo, j_lo = model.both(x4)
    sd_lo, j_lo = sd_lo.float(), j_lo.float()
    joint = lambda t: F.cross_entropy(j_lo, t.repeat(4) * 4 + rot.repeat_interleave(B))
    l_sla = lam * joint(ya) + (1 - lam) * joint(yb)                               # joint (class, rotation) labels
    agg = torch.stack([j_lo.view(4, B, 10, 4)[k, :, :, k] for k in range(4)]).mean(0)   # aggregated class logits
    sd0 = sd_lo[:B]                                                               # 10-class head, upright image
    l_kl = F.kl_div(F.log_softmax(sd0, 1), F.softmax(agg.detach(), 1), reduction="batchmean")
    l_ce = lam * F.cross_entropy(sd0, ya) + (1 - lam) * F.cross_entropy(sd0, yb)   # beta term
    return sd0, l_sla + l_kl + CFG["BETA"] * l_ce

for epoch in range(start, CFG["EPOCHS"]):
    t0 = time.time(); model.train()
    tot, correct = torch.zeros((), device=device), torch.zeros((), device=device)
    perm = torch.randperm(len(Xtr), device=device)
    for i in range(0, len(Xtr), CFG["BATCH_SIZE"]):
        idx = perm[i:i + CFG["BATCH_SIZE"]]; B = len(idx)
        x, ya, yb, lam = augment(Xtr[idx]), ytr[idx], ytr[idx], 1.0
        if np.random.rand() < CFG["MIX_PROB"]:
            x, ya, yb, lam = resizemix(x, ya)
        x4 = torch.cat([torch.rot90(x, k, (2, 3)) for k in range(4)]).contiguous(memory_format=torch.channels_last)

        # ASAM (Kwon et al. 2021): gradient at the worst nearby weights, perturbation scaled by weight size
        opt.zero_grad(set_to_none=True)
        out, loss = loss_fn(x4, ya, yb, lam, B); loss.backward()
        with torch.no_grad():
            tw = [p.abs() + 0.01 for p in params]
            norm = torch.norm(torch.stack([(t * p.grad).norm() for t, p in zip(tw, params) if p.grad is not None])) + 1e-12
            eps = [t * t * p.grad * (CFG["ASAM_RHO"] / norm) if p.grad is not None else None for t, p in zip(tw, params)]
            for p, e in zip(params, eps):
                if e is not None: p.add_(e)
        opt.zero_grad(set_to_none=True)
        loss_fn(x4, ya, yb, lam, B)[1].backward()
        with torch.no_grad():
            for p, e in zip(params, eps):
                if e is not None: p.sub_(e)
        opt.step(); sched.step(); ema.update_parameters(model)
        tot += loss.detach() * B; correct += (out.argmax(1) == ya).sum()

    val_acc = acc(predict(ema.module, Xva, tta=False), yva)
    history.append({"epoch": epoch + 1, "loss": tot.item() / len(Xtr), "train_acc": correct.item() / len(Xtr), "val_acc": val_acc,
                    "lr": sched.get_last_lr()[0], "sec": time.time() - t0})
    if val_acc > best_acc:
        best_acc = val_acc
        save({"model": ema.module.state_dict(), "cfg": CFG, "epoch": epoch + 1, "val_acc": val_acc,
              "norm": {"mean": MEAN.cpu(), "std": STD.cpu()}}, BEST)
    save({"model": model.state_dict(), "ema": ema.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
          "epoch": epoch + 1, "best_acc": best_acc, "history": history, "cfg": CFG}, LAST)
    h = history[-1]
    print(f"ep {epoch+1:3d}/{CFG['EPOCHS']} | loss {h['loss']:.4f} | train acc {h['train_acc']:.4f} | val (EMA) {val_acc:.4f}"
          f"{' *' if best_acc == val_acc else '  '} | lr {h['lr']:.4f} | {h['sec']:.0f}s | eta {(CFG['EPOCHS']-epoch-1)*h['sec']/3600:.1f} h")
pd.DataFrame(history).to_csv(os.path.join(RUN_DIR, "history.csv"), index=False)
print("done, best val", best_acc)

ep   1/200 | loss 5.7513 | train acc 0.2014 | val (EMA) 0.1108 * | lr 0.0416 | 331s | eta 18.3 h
ep   2/200 | loss 5.4933 | train acc 0.2449 | val (EMA) 0.1528 * | lr 0.0812 | 278s | eta 15.3 h
ep   3/200 | loss 5.2518 | train acc 0.2857 | val (EMA) 0.1772 * | lr 0.1208 | 278s | eta 15.2 h
ep   4/200 | loss 5.0158 | train acc 0.3223 | val (EMA) 0.2292 * | lr 0.1604 | 278s | eta 15.1 h
ep   5/200 | loss 4.7615 | train acc 0.3738 | val (EMA) 0.2272   | lr 0.2000 | 278s | eta 15.1 h
ep   6/200 | loss 4.4956 | train acc 0.4088 | val (EMA) 0.2340 * | lr 0.2000 | 278s | eta 15.0 h
ep   7/200 | loss 4.2006 | train acc 0.4620 | val (EMA) 0.2388 * | lr 0.1999 | 278s | eta 14.9 h
ep   8/200 | loss 3.9999 | train acc 0.4817 | val (EMA) 0.2504 * | lr 0.1999 | 278s | eta 14.8 h
ep   9/200 | loss 3.8391 | train acc 0.5110 | val (EMA) 0.2632 * | lr 0.1998 | 278s | eta 14.8 h
ep  10/200 | loss 3.7058 | train acc 0.5338 | val (EMA) 0.2724 * | lr 0.1997 | 278s | eta 14.7 h
ep  11/200 | loss 3.4594 | tra

KeyboardInterrupt: 

## 8. Evaluate best.pt (pick SD / AG / SD+AG on validation only)

In [8]:
best = torch.load(BEST, map_location=device, weights_only=False)
net = ResNeXt29(CFG["CARDINALITY"], CFG["BOTTLENECK_WIDTH"], CFG["WIDTHS"], CFG["BLOCKS"]).to(device).to(memory_format=torch.channels_last)
net.load_state_dict(best["model"]); net.eval()
print(f"best.pt from epoch {best['epoch']}")

val_accs = {m: acc(predict(net, Xva, m), yva) for m in ("sd", "ag", "sdag")}
alt = max(("ag", "sdag"), key=val_accs.get)
MODE = alt if val_accs[alt] > val_accs["sd"] + 0.002 else "sd"        # SD unless another mode wins by 5+ images
print("val:", {k: round(v, 4) for k, v in val_accs.items()}, "-> using", MODE)

p_te = predict(net, Xte, MODE)
print(f"INTERNAL TEST accuracy: {acc(p_te, yte):.4f}   (0.90 Kaggle run: 0.9760)")
pred = p_te.argmax(1)
cm = torch.bincount(yte * 10 + pred, minlength=100).view(10, 10).cpu()
print(pd.DataFrame({"recall": (cm.diag() / cm.sum(1)).numpy().round(3), "errors": (cm.sum(1) - cm.diag()).numpy()}, index=CLASSES).T.to_string())
off = cm.clone(); off.fill_diagonal_(0)
top = sorted(((int(off[i, j]), CLASSES[i], CLASSES[j]) for i in range(10) for j in range(10) if off[i, j] > 0), reverse=True)[:6]
print("top confusions (true -> predicted):", ", ".join(f"{n} {a}->{b}" for n, a, b in top))

best.pt from epoch 172
val: {'sd': 0.9756, 'ag': 0.9744, 'sdag': 0.9764} -> using sd
INTERNAL TEST accuracy: 0.9752   (0.90 Kaggle run: 0.9760)
        airplane  automobile    bird    cat   deer     dog   frog  horse   ship  truck
recall     0.972       0.984   0.952   0.94  0.988   0.948  0.996   0.98  0.992    1.0
errors     7.000       4.000  12.000  15.00  3.000  13.000  1.000   5.00  2.000    0.0
top confusions (true -> predicted): 10 cat->dog, 9 dog->cat, 5 cat->bird, 5 bird->cat, 4 automobile->truck, 3 bird->dog


## 9. Kaggle submission (one network)

In [9]:
labels = predict(net, Xkg, MODE).argmax(1).cpu().numpy()
sub = pd.DataFrame({"id": kaggle_ids.astype(int), "label": labels.astype(int)})
assert list(sub.columns) == ["id", "label"] and len(sub) == len(sample_sub)
assert (sub["id"].values == sample_sub["id"].values).all() and sub["label"].between(0, 9).all()
path = os.path.join(RUN_DIR, f"submission_{MODE}_test{acc(p_te, yte):.4f}.csv")
sub.to_csv(path, index=False); sub.to_csv("/content/submission.csv", index=False)
print("saved", path, "| predicted counts:", np.bincount(labels, minlength=10).tolist())
from google.colab import files; files.download("/content/submission.csv")

saved /content/drive/MyDrive/cifar10_resnet_challenge/runs/resnext29_slasd_plainCE_beta1_resizemix_v1/submission_sd_test0.9752.csv | predicted counts: [167, 210, 207, 194, 198, 218, 188, 204, 210, 204]


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import runtime
runtime.unassign()
